# Day 55 Practice: Prototype the Project's Hard Parts

Use this notebook as a **workbench**: get each piece working here on small examples, then copy the function into
your project's `app.py` and run `uv run pytest -q` there.

✅ Exercises 1–3: everyone, in class · Exercise 4: before you write `analyse_all` (homework) · Exercise 5: challenge

In [ ]:
import csv
import json
from pathlib import Path

from llm import mock_llm

with open(Path("../../../datasets/reviews.csv"), encoding="utf-8", newline="") as file:
    reviews = list(csv.DictReader(file))
print(len(reviews), "reviews loaded")

## Exercise 1: Know your data

a) Print the number of reviews and the average rating for each product.
b) Make a list of the IDs of all reviews whose text contains an `@` or any digit.

Expected output:
```
wireless earbuds: 8 reviews, average 3.4
coffee maker: 8 reviews, average 3.4
backpack: 8 reviews, average 3.4
desk lamp: 8 reviews, average 3.2
running shoes: 8 reviews, average 3.2
Reviews with @ or digits: ['R21', 'R23', 'R33']
```
Which of those three reviews really contain personal data?

In [ ]:
# your code here

*Your answer:*

## Exercise 2: Redaction prototype

Write three functions:
- `looks_like_email(word)`: `True` if the word has an `@` with text before it and a `.` somewhere after it;
- `looks_like_phone(word)`: `True` if the word has **at least 7 digits** and digits make up at least 70% of it;
- `redact(text)`: goes word by word, strips punctuation around each word (`word.strip(PUNCTUATION)`), and replaces
  only that core with `[EMAIL]` or `[PHONE]`, so the punctuation stays.

All six checks must print `True`.

In [ ]:
PUNCTUATION = ".,!?;:()\"'"

# your code here


cases = [
    ("Call me at 555-0142.", "Call me at [PHONE]."),
    ("Mail jo.example@example.com today", "Mail [EMAIL] today"),
    ("No blisters after 10 km.", "No blisters after 10 km."),
    ("Ring +44-20-7946-0958 (evenings)", "Ring [PHONE] (evenings)"),
    ("Price was 49.99 for 2 items", "Price was 49.99 for 2 items"),
    ("Contact: mei@shop.example, 0300-1234567!", "Contact: [EMAIL], [PHONE]!"),
]
# for text, expected in cases:
#     print(redact(text) == expected)

Name one piece of personal data your rule **misses**, and one harmless word it would **wrongly** hide.

*Your answer:*

## Exercise 3: One review, end to end

1. Write `build_prompt(product, text)`: the Day 52/54 ingredients (role, task, allowed labels, "only a JSON object"
   with the keys `sentiment`, `topics`, `summary`, `<review>` tags) and **`redact(text)` inside the tags**.
2. For R33: print the prompt; check that it contains `[PHONE]` and does **not** contain `555-0142`.
3. Send it to `mock_llm`, cut out the `{...}`, `json.loads` it and print the dict.

Expected output (after your printed prompt):
```
True True
{'sentiment': 'negative', 'topics': ['support'], 'summary': 'Please call me at [PHONE] or email [EMAIL] about my refund for this backpack.'}
```

In [ ]:
r33 = reviews[32]

# your code here

## Exercise 4: Keep going, keep count

Here are five replies a model might give. Write `parse_response(reply)` (cut out the `{...}`, `json.loads`, check
the keys, an allowed sentiment, topics is a list, summary is non-empty text; raise `ValueError` otherwise) and a
loop that **never crashes**, prints one line per reply and counts the successes and failures.

Expected output (your error messages may be worded differently):
```
reply 1: ok, positive
reply 2: ok, negative
reply 3: failed (invalid sentiment: neutral)
reply 4: failed (no JSON object in the reply)
reply 5: failed (topics must be a list)
ok: 2, failed: 3
```

In [ ]:
ALLOWED_SENTIMENTS = ["positive", "negative", "mixed"]
replies = [
    '{"sentiment": "positive", "topics": ["sound"], "summary": "Great sound."}',
    'Here you go:\n```json\n{"sentiment": "negative", "topics": ["battery"], "summary": "Dies fast."}\n```',
    '{"sentiment": "neutral", "topics": [], "summary": "It is fine."}',
    "Sorry, I cannot help with that.",
    '{"sentiment": "mixed", "topics": "price", "summary": "Good but pricey."}',
]

# your code here

## Exercise 5 (challenge): Stars vs text

a) Write `find_disagreements(results)`: return the results with 4–5 stars but a negative sentiment, or 1–2 stars but
a positive sentiment. Test it on the list below (expected: `['R36', 'X01']`).

b) Use your functions from Exercises 3 and 4 on **all 40 reviews** with `mock_llm`, then run `find_disagreements`
on the results. Expected: `['R36']`. Read R36: is it the stars or the text that tells the truth?

In [ ]:
sample_results = [
    {"review_id": "R01", "rating": 5, "sentiment": "positive"},
    {"review_id": "R36", "rating": 5, "sentiment": "negative"},
    {"review_id": "R17", "rating": 1, "sentiment": "negative"},
    {"review_id": "R35", "rating": 2, "sentiment": "mixed"},
    {"review_id": "R40", "rating": 4, "sentiment": "mixed"},
    {"review_id": "X01", "rating": 1, "sentiment": "positive"},
]

# your code here